# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
print("Token added successfully:", hf_token is not None)
file_c = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token)
file_f = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token)
df_c = pd.read_parquet(file_c)
df_f = pd.read_parquet(file_f)
print(df_c.shape, df_f.shape)
df_c.head()

Token added successfully: True
(519606, 26) (9841378, 30)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit Of Analysis:** 1 row = 1 unique webpage (content_hash_id), aggregated over March 2026
* **Time Window:** month=2026-03, loaded directly from the warehouse partition (verified below)
* **Table Used:** fact_content_daily_performance (month=2026-03 partition) + dim_content, from the HF warehouse
* **Predict / Rank:** opportunity_flags (0-3 count of risk conditions: declining, page-one, stale)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_f['report_date'] = pd.to_datetime(df_f['report_date'])
print("Date range in this partition:", df_f['report_date'].min(), "to", df_f['report_date'].max())
print("Confirms this is a true March 2026 slice, not the full warehouse.")

Date range in this partition: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Confirms this is a true March 2026 slice, not the full warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.

* **Features:** avg_position, total_clicks, total_impressions, click_trend — all knowable by end of March 2026, all raw aggregates from fact_content_daily_performance, none derived from the label.
* **Label:** opportunity_flags — count of true risk conditions (declining, page-one, [stale — pending]), transparent, no invented weights.
* **Context:** content_hash_id (page identifier, not predictive).
* **Excluded:** raw URLs, client identifiers — anonymization/privacy. cpc, search_volume, word_count — not present in the real warehouse tables, were starter-CSV-only simplifications.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
mid = df_f['report_date'].median()
page_stats = df_f.groupby('content_hash_id').agg(
    total_clicks=('gsc_clicks', 'sum'),
    total_impressions=('gsc_impressions', 'sum'),
    avg_position=('gsc_avg_position', 'mean')).reset_index()
early_clicks = df_f[df_f['report_date'] < mid].groupby('content_hash_id')['gsc_clicks'].sum()
late_clicks = df_f[df_f['report_date'] >= mid].groupby('content_hash_id')['gsc_clicks'].sum()
click_trend = (late_clicks - early_clicks).rename('click_trend')
df = page_stats.merge(click_trend, on='content_hash_id', how='left')
print(df.shape)
df.head()
df['is_declining'] = (df['click_trend'] < 0).astype(int)
df['is_page_one'] = (df['avg_position'] <= 10).astype(int)
df['opportunity_flags'] = df['is_declining'] + df['is_page_one']
features = ['avg_position', 'total_clicks', 'total_impressions', 'click_trend']
label = ['opportunity_flags']
context = ['content_hash_id']
print(f"Features ({len(features)}): {features}")
print(f"Label (1): {label}")

(331437, 5)
Features (4): ['avg_position', 'total_clicks', 'total_impressions', 'click_trend']
Label (1): ['opportunity_flags']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Time Window: This starter snapshot (content_refresh_anonymized.csv) has no date/month column — it's a single static cross-section, not time-partitioned. A real time window (e.g. month=2026-03) will apply once we move to the Hugging Face warehouse release in Week 3.

* **Grain Check:** Confirming zero duplicate content_id entries.
* **Row Count & Completeness:** Measuring overall row count and non-null availability across features.
* **Availability Filter (IS TRUE check):** Verifying how many rows survive when requiring valid, non-null feature values.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dup=df.duplicated(subset=['content_id']).sum()
print(f"The duplicate grains are {dup}")
print(f"\nThe total rows are {len(df):,}")
print("\nChecking null values in features")
print(df[features].notnull().sum())
is_complete_row = df[features].notnull().all(axis=1) & df['content_id'].notnull()
surviving_rows = is_complete_row.sum()
survival_pct = (surviving_rows / total_rows) * 100
print("\nAVAILABILITY FILTER (IS TRUE)")
print(f"Feature columns checked: {features}")
print(f"Surviving rows after filter: {surviving_rows:,} / {total_rows:,} ({survival_pct:.1f}% survived)")

The duplicate grains are 0

The total rows are 30,000

Checking null values in features
search_volume    27532
avg_position     30000
cpc              27532
word_count       22301
trend_pct        26612
dtype: int64

AVAILABILITY FILTER (IS TRUE)
Feature columns checked: ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
Surviving rows after filter: 18,013 / 30,000 (60.0% survived)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What can this data never tell you?**

This dataset cannot observe external search engine algorithm updates or competitor backlink changes. A drop in traffic might be driven by external core updates rather than content staleness.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['cheat'] = df['opportunity_score'] * 0.99
print("Cheat Correlation:", df['cheat'].corr(df['opportunity_score']))
df.drop(columns=['cheat'], inplace=True)
print("cheat column is removed")

Cheat Correlation: 1.0
cheat column is removed


## Self-check

Before you submit, confirm each line honestly:

- [ Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes ] No client names, URLs, or private queries anywhere
- [ Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.